In [7]:
import gzip
import math
import pandas as pd
import numpy as np
from collections import Counter

file_path = "/Users/newlanr1/Downloads/clean_pairs_annotated.csv.gz"  # update path if needed
target_col = "read_count"                   # update column if needed

# 1. Stream & aggregate counts
chunk_size = 100000
freq_counter = Counter()
total_unique_clones = 0
total_reads = 0

print(f"Processing '{target_col}' in chunks...")

for chunk in pd.read_csv(file_path, compression="gzip", usecols=[target_col], chunksize=chunk_size):
    counts = chunk[target_col].dropna().astype(int)
    total_unique_clones += len(counts)
    total_reads += counts.sum()
    freq_counter.update(counts)

f1 = freq_counter.get(1, 0)  # Singletons
f2 = freq_counter.get(2, 0)  # Doubletons

N = float(total_reads)
S_obs = float(total_unique_clones)

print("\n=== DATASET SUMMARY ===")
print(f"Total Reads (N)         : {int(N):,}")
print(f"Unique Clones (S_obs)   : {int(S_obs):,}")
print(f"Singletons (f1, count=1): {f1:,}")
print(f"Doubletons (f2, count=2): {f2:,}")

# ----------------------------------------------------------------------
# 2. HELPER FUNCTIONS FOR MULTI-LEVEL CIs
# ----------------------------------------------------------------------
# Standard z-scores for common coverage levels
Z_SCORES = {
    "80%": 1.282,
    "90%": 1.645,
    "95%": 1.960,
    "99%": 2.576
}

def chao1_multi_ci(S_obs, f1, f2):
    """Calculates Chao1 estimate and CIs across multiple alpha levels."""
    if f2 > 0 and f1 > 0:
        f1_f2_ratio = f1 / f2
        f1_sq = f1 ** 2
        f_hat = f1_sq / (2.0 * f2)
        S_chao1 = S_obs + f_hat
        var_S = f2 * (((f1_f2_ratio / 4.0) ** 4) + ((f1_f2_ratio) ** 3) + ((f1_f2_ratio / 2.0) ** 2))
    else:
        return S_obs, {level: (S_obs, S_obs) for level in Z_SCORES}

    cis = {}
    for level, z in Z_SCORES.items():
        if var_S > 0 and f_hat > 0:
            C_factor = math.exp(z * math.sqrt(math.log(1.0 + (var_S / (f_hat ** 2)))))
            ci_lower = S_obs + (f_hat / C_factor)
            ci_upper = S_obs + (f_hat * C_factor)
        else:
            ci_lower, ci_upper = S_chao1, S_chao1
        cis[level] = (ci_lower, ci_upper)
        
    return S_chao1, cis

def poisson_solver(N, k):
    """Solves N/C = -ln(1 - k/C) for total diversity C."""
    if N <= k or k <= 0: return float('inf')
    low, high = float(k), 1e15
    for _ in range(200):
        mid = (low + high) / 2.0
        if mid * (1.0 - math.exp(-N / mid)) < k:
            low = mid
        else:
            high = mid
    return (low + high) / 2.0

def poisson_multi_ci(N, k):
    """Calculates Poisson estimate and CIs across multiple alpha levels."""
    S_poisson = poisson_solver(N, k)
    duplicates = N - k
    
    cis = {}
    if duplicates > 0:
        se_dup = math.sqrt(duplicates)
        for level, z in Z_SCORES.items():
            k_upper = N - max(1.0, duplicates - z * se_dup)
            k_lower = N - (duplicates + z * se_dup)
            ci_lower = poisson_solver(N, k_lower)
            ci_upper = poisson_solver(N, k_upper)
            cis[level] = (ci_lower, ci_upper)
    else:
        cis = {level: (S_poisson, S_poisson) for level in Z_SCORES}
        
    return S_poisson, cis

# ----------------------------------------------------------------------
# 3. COMPUTE & DISPLAY MULTI-LEVEL CIs
# ----------------------------------------------------------------------
s_chao, chao_cis = chao1_multi_ci(S_obs, f1, f2)
s_pois, pois_cis = poisson_multi_ci(N, S_obs)

# Build summary DataFrame
records = []
for level in Z_SCORES.keys():
    records.append({
        "Confidence Level": level,
        "Chao1 Lower CI": f"{chao_cis[level][0]:,.0f}",
        "Chao1 Upper CI": f"{chao_cis[level][1]:,.0f}",
        "Poisson Lower CI": f"{pois_cis[level][0]:,.0f}",
        "Poisson Upper CI": f"{pois_cis[level][1]:,.0f}",
    })

df_ci = pd.DataFrame(records)

print(f"\n=== POINT ESTIMATES ===")
print(f"Chao1 Estimate : {s_chao:,.0f} unique clones")
print(f"Poisson Estimate: {s_pois:,.0f} unique clones\n")

print("=== CONFIDENCE INTERVAL TABLE ===")
print(df_ci.to_string(index=False))

Processing 'read_count' in chunks...

=== DATASET SUMMARY ===
Total Reads (N)         : 3,702,377
Unique Clones (S_obs)   : 3,659,983
Singletons (f1, count=1): 3,617,839
Doubletons (f2, count=2): 41,896

=== POINT ESTIMATES ===
Chao1 Estimate : 159,865,337 unique clones
Poisson Estimate: 160,432,555 unique clones

=== CONFIDENCE INTERVAL TABLE ===
Confidence Level Chao1 Lower CI Chao1 Upper CI Poisson Lower CI Poisson Upper CI
             80%    159,621,760    160,109,295      159,432,156      161,445,490
             90%    159,552,860    160,178,441      159,151,136      161,734,614
             95%    159,493,095    160,238,469      158,908,072      161,986,341
             99%    159,376,288    160,355,923      158,434,869      162,480,860
